In [ ]:
# Import all necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc, 
    precision_recall_curve, roc_auc_score, accuracy_score,
    precision_score, recall_score, f1_score
)
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("📦 All libraries imported successfully\!")
print("🎨 Visualization style configured\!")

## 🔍 Understanding TP, FP, TN, and FN

All classification metrics are built upon four fundamental concepts:

| Prediction vs Reality | Predicted Positive | Predicted Negative |
|---------------------|-------------------|-------------------|
| **Actually Positive** | True Positive (TP) | False Negative (FN) |
| **Actually Negative** | False Positive (FP) | True Negative (TN) |

### 📝 Definitions

- **True Positive (TP)**: Correctly identified positive cases
- **True Negative (TN)**: Correctly identified negative cases
- **False Positive (FP)**: Incorrectly identified as positive (Type I error)
- **False Negative (FN)**: Incorrectly identified as negative (Type II error)

### 🏥 Medical Example

In cancer detection:
- **TP**: Model predicts cancer, patient has cancer ✅
- **TN**: Model predicts no cancer, patient is healthy ✅
- **FP**: Model predicts cancer, patient is healthy ❌ (unnecessary worry/treatment)
- **FN**: Model predicts no cancer, patient has cancer ❌ (missed diagnosis - dangerous\!)

# 📊 Classification Metrics: Comprehensive Guide

This notebook provides a thorough exploration of classification metrics, from basic concepts to advanced applications. 
We'll cover:

1. **Fundamental Concepts**: TP, FP, TN, FN and the Confusion Matrix
2. **Core Metrics**: Accuracy, Precision, Recall, F1-Score, Specificity
3. **Advanced Metrics**: ROC-AUC, PR-AUC, Matthews Correlation Coefficient
4. **Practical Applications**: Model comparison, threshold optimization
5. **Interactive Visualizations**: Comprehensive dashboards and comparisons

## 🎯 Learning Objectives

By the end of this notebook, you will:
- Understand all major classification metrics and when to use them
- Be able to implement metrics calculations from scratch
- Know how to create comprehensive evaluation dashboards
- Understand the trade-offs between different metrics
- Be able to optimize model performance based on specific requirements

In [ ]:
class ClassificationMetrics:
    """
    A comprehensive class for calculating and visualizing classification metrics.
    
    This class provides methods to:
    - Calculate all standard classification metrics
    - Generate visualizations (confusion matrix, ROC curve, PR curve)
    - Compare multiple models
    - Analyze threshold optimization
    """
    
    def __init__(self):
        self.metrics_definitions = {
            'Accuracy': 'Overall correctness: (TP + TN) / (TP + TN + FP + FN)',
            'Precision': 'Positive predictive value: TP / (TP + FP)',
            'Recall': 'Sensitivity, true positive rate: TP / (TP + FN)',
            'Specificity': 'True negative rate: TN / (TN + FP)',
            'F1_Score': 'Harmonic mean of precision and recall: 2TP / (2TP + FP + FN)',
            'AUC': 'Area under ROC curve: probability model ranks random positive > random negative',
            'MCC': 'Matthews Correlation Coefficient: correlation between observed and predicted'
        }
    
    def safe_divide(self, numerator, denominator):
        """Safe division that handles division by zero."""
        return numerator / denominator if denominator != 0 else 0
    
    def calculate_all_metrics(self, y_true, y_pred, y_prob=None):
        """
        Calculate comprehensive classification metrics.
        
        Parameters:
        -----------
        y_true : array-like
            True binary labels
        y_pred : array-like  
            Predicted binary labels
        y_prob : array-like, optional
            Predicted probabilities for positive class
            
        Returns:
        --------
        dict : Dictionary containing all calculated metrics
        """
        # Get confusion matrix components
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        
        metrics = {
            # Basic counts
            'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
            
            # Primary metrics
            'Accuracy': self.safe_divide(tp + tn, tp + tn + fp + fn),
            'Precision': self.safe_divide(tp, tp + fp),
            'Recall': self.safe_divide(tp, tp + fn),
            'Specificity': self.safe_divide(tn, tn + fp),
            'F1_Score': self.safe_divide(2 * tp, 2 * tp + fp + fn),
            
            # Additional metrics
            'Balanced_Accuracy': (self.safe_divide(tp, tp + fn) + self.safe_divide(tn, tn + fp)) / 2,
            'False_Positive_Rate': self.safe_divide(fp, fp + tn),
            'False_Discovery_Rate': self.safe_divide(fp, tp + fp),
            'Negative_Predictive_Value': self.safe_divide(tn, tn + fn),
            
            # Advanced metrics
            'Matthews_Correlation_Coefficient': self.calculate_mcc(tp, tn, fp, fn),
        }
        
        # Add AUC if probabilities are provided
        if y_prob is not None:
            try:
                metrics['AUC'] = roc_auc_score(y_true, y_prob)
            except ValueError:
                metrics['AUC'] = 0.5  # Default for edge cases
                
        return metrics
    
    def calculate_mcc(self, tp, tn, fp, fn):
        """Calculate Matthews Correlation Coefficient."""
        numerator = tp * tn - fp * fn
        denominator = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
        return self.safe_divide(numerator, denominator)
    
    def plot_confusion_matrix(self, y_true, y_pred, title="Confusion Matrix", labels=None):
        """
        Plot an enhanced confusion matrix with annotations and percentages.
        """
        cm = confusion_matrix(y_true, y_pred)
        
        # Calculate percentages
        cm_percent = cm.astype('float') / cm.sum() * 100
        
        fig, ax = plt.subplots(figsize=(8, 6))
        
        # Create heatmap
        sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', 
                   cbar_kws={'label': 'Number of Predictions'})
        
        # Add custom annotations with counts and percentages
        thresh = cm.max() / 2.
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j + 0.5, i + 0.5, f'{cm[i, j]}\\n({cm_percent[i, j]:.1f}%)',
                       ha="center", va="center",
                       color="white" if cm[i, j] > thresh else "black",
                       fontsize=12, fontweight='bold')
        
        ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
        ax.set_ylabel('True Label', fontsize=12)
        ax.set_xlabel('Predicted Label', fontsize=12)
        
        if labels is None:
            labels = ['Negative', 'Positive']
        ax.set_xticklabels(labels)
        ax.set_yticklabels(labels)
        
        plt.tight_layout()
        return fig, ax
    
    def create_metrics_dashboard(self, y_true, y_pred, y_prob=None, title="Classification Metrics Dashboard"):
        """
        Create a comprehensive dashboard with multiple visualizations.
        """
        metrics = self.calculate_all_metrics(y_true, y_pred, y_prob)
        
        fig = plt.figure(figsize=(16, 12))
        
        # Confusion Matrix
        ax1 = plt.subplot(2, 3, 1)
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1)
        ax1.set_title('Confusion Matrix')
        ax1.set_ylabel('True Label')
        ax1.set_xlabel('Predicted Label')
        
        # Metrics Bar Chart
        ax2 = plt.subplot(2, 3, 2)
        metric_names = ['Accuracy', 'Precision', 'Recall', 'F1_Score', 'Specificity']
        metric_values = [metrics[name] for name in metric_names]
        bars = ax2.bar(metric_names, metric_values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
        ax2.set_title('Key Metrics')
        ax2.set_ylim(0, 1)
        ax2.tick_params(axis='x', rotation=45)
        
        # Add value labels on bars
        for bar, value in zip(bars, metric_values):
            ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                    f'{value:.3f}', ha='center', va='bottom')
        
        if y_prob is not None:
            # ROC Curve
            ax3 = plt.subplot(2, 3, 3)
            fpr, tpr, _ = roc_curve(y_true, y_prob)
            roc_auc = auc(fpr, tpr)
            ax3.plot(fpr, tpr, label=f'ROC (AUC = {roc_auc:.3f})')
            ax3.plot([0, 1], [0, 1], 'k--', label='Random')
            ax3.set_xlabel('False Positive Rate')
            ax3.set_ylabel('True Positive Rate')
            ax3.set_title('ROC Curve')
            ax3.legend()
            ax3.grid(True, alpha=0.3)
            
            # Precision-Recall Curve
            ax4 = plt.subplot(2, 3, 4)
            precision, recall, _ = precision_recall_curve(y_true, y_prob)
            ap = np.trapz(precision, recall)
            ax4.plot(recall, precision, label=f'PR (AP = {ap:.3f})')
            baseline = np.sum(y_true) / len(y_true)
            ax4.axhline(y=baseline, color='r', linestyle='--', label=f'Baseline ({baseline:.3f})')
            ax4.set_xlabel('Recall')
            ax4.set_ylabel('Precision')
            ax4.set_title('Precision-Recall Curve')
            ax4.legend()
            ax4.grid(True, alpha=0.3)
        
        # Metrics Summary Table
        ax5 = plt.subplot(2, 3, 5)
        ax5.axis('tight')
        ax5.axis('off')
        
        table_data = []
        for key, value in metrics.items():
            if isinstance(value, (int, float)) and key not in ['TP', 'TN', 'FP', 'FN']:
                table_data.append([key.replace('_', ' '), f'{value:.4f}'])
        
        table = ax5.table(cellText=table_data, colLabels=['Metric', 'Value'],
                         cellLoc='left', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1.2, 1.5)
        ax5.set_title('Detailed Metrics')
        
        plt.suptitle(title, fontsize=16, fontweight='bold', y=0.98)
        plt.tight_layout()
        return fig

# Initialize the metrics calculator
calc = ClassificationMetrics()
print("🧮 ClassificationMetrics class initialized successfully!")
print("📚 Available methods:")
print("  • calculate_all_metrics() - Calculate all metrics")
print("  • plot_confusion_matrix() - Enhanced confusion matrix") 
print("  • create_metrics_dashboard() - Comprehensive dashboard")
print("  • And more...")

## 🎯 Practical Demonstration

Let's apply our ClassificationMetrics class to a real dataset. We'll use the breast cancer dataset to demonstrate:

1. **Data Loading & Preparation**
2. **Model Training**
3. **Comprehensive Metrics Calculation**
4. **Interactive Visualizations**

### Dataset: Breast Cancer Classification
- **Objective**: Predict whether a tumor is malignant (1) or benign (0)
- **Features**: 30 numerical features describing cell nuclei characteristics
- **Classes**: Balanced dataset with both malignant and benign cases

In [ ]:
# 📊 STEP 1: DATA LOADING AND PREPARATION
print("📊 STEP 1: Data Loading and Preparation")
print("=" * 50)

# Load the breast cancer dataset
data = load_breast_cancer()
X, y = data.data, data.target

print(f"Dataset shape: {X.shape}")
print(f"Target classes: {np.unique(y)} (0=malignant, 1=benign)")
print(f"Class distribution: {np.bincount(y)}")
print(f"Positive class ratio: {np.mean(y):.3f}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features (important for logistic regression and SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\\nTrain set: {X_train_scaled.shape[0]} samples")
print(f"Test set: {X_test_scaled.shape[0]} samples")
print("✅ Data preparation completed!")

In [ ]:
# 🤖 STEP 2: MODEL TRAINING AND PREDICTIONS
print("🤖 STEP 2: Model Training and Predictions")
print("=" * 50)

# Train a logistic regression model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# Get predictions and probabilities
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]  # Probability of positive class

print(f"Model trained successfully!")
print(f"Predictions shape: {y_pred.shape}")
print(f"Probabilities shape: {y_prob.shape}")
print(f"Prediction range: {y_pred.min()} to {y_pred.max()}")
print(f"Probability range: {y_prob.min():.3f} to {y_prob.max():.3f}")
print("✅ Model training completed!")

In [ ]:
# 📈 STEP 3: COMPREHENSIVE METRICS CALCULATION
print("📈 STEP 3: Comprehensive Metrics Calculation")
print("=" * 50)

# Calculate all metrics using our custom class
metrics = calc.calculate_all_metrics(y_test, y_pred, y_prob)

print("🏆 DETAILED METRICS RESULTS:")
print("-" * 40)

# Display basic counts
print("📊 Confusion Matrix Components:")
print(f"  True Positives (TP):  {metrics['TP']:3d}")
print(f"  True Negatives (TN):  {metrics['TN']:3d}")
print(f"  False Positives (FP): {metrics['FP']:3d}")
print(f"  False Negatives (FN): {metrics['FN']:3d}")

print("\\n🎯 Primary Metrics:")
print(f"  Accuracy:     {metrics['Accuracy']:.4f}")
print(f"  Precision:    {metrics['Precision']:.4f}")
print(f"  Recall:       {metrics['Recall']:.4f}")
print(f"  Specificity:  {metrics['Specificity']:.4f}")
print(f"  F1-Score:     {metrics['F1_Score']:.4f}")

print("\\n🔍 Advanced Metrics:")
print(f"  AUC:                           {metrics['AUC']:.4f}")
print(f"  Balanced Accuracy:             {metrics['Balanced_Accuracy']:.4f}")
print(f"  Matthews Correlation Coeff:    {metrics['Matthews_Correlation_Coefficient']:.4f}")
print(f"  False Positive Rate:           {metrics['False_Positive_Rate']:.4f}")
print(f"  Negative Predictive Value:     {metrics['Negative_Predictive_Value']:.4f}")

print("\\n💡 INTERPRETATION:")
print(f"  • Model correctly classifies {metrics['Accuracy']*100:.1f}% of cases")
print(f"  • When predicting positive, it's correct {metrics['Precision']*100:.1f}% of the time")
print(f"  • It catches {metrics['Recall']*100:.1f}% of actual positive cases")
print(f"  • AUC of {metrics['AUC']:.3f} indicates {'excellent' if metrics['AUC'] > 0.9 else 'good' if metrics['AUC'] > 0.8 else 'fair'} discrimination")

In [ ]:
# 🎨 STEP 4: INTERACTIVE VISUALIZATIONS
print("🎨 STEP 4: Creating Interactive Visualizations")
print("=" * 50)

# Create enhanced confusion matrix
print("Creating enhanced confusion matrix...")
calc.plot_confusion_matrix(y_test, y_pred, 
                          title="🏥 Breast Cancer Classification - Confusion Matrix",
                          labels=['Malignant', 'Benign'])
plt.show()

print("\\n✅ Enhanced confusion matrix created!")

In [ ]:
# 📊 COMPREHENSIVE METRICS DASHBOARD
print("📊 Creating Comprehensive Metrics Dashboard")
print("=" * 50)

# Create the complete dashboard with all visualizations
dashboard_fig = calc.create_metrics_dashboard(
    y_test, y_pred, y_prob,
    title="🏥 Breast Cancer Classification - Complete Dashboard"
)
plt.show()

print("✅ Complete metrics dashboard created!")
print("\\n🎉 The dashboard includes:")
print("  • Confusion Matrix with counts")
print("  • Key Metrics Bar Chart")
print("  • ROC Curve with AUC")
print("  • Precision-Recall Curve")
print("  • Detailed Metrics Table")

## 🎓 Key Takeaways and Best Practices

### ✅ **What We've Learned:**

1. **Foundation**: All classification metrics derive from TP, TN, FP, FN
2. **Metric Selection**: Choose metrics based on your problem's characteristics
3. **Visualization**: Comprehensive dashboards provide better insights than single numbers
4. **Interpretation**: Understanding what each metric tells you is crucial

### 📊 **Metric Selection Guide:**

| **Scenario** | **Primary Metrics** | **Why** |
|--------------|-------------------|---------|
| Balanced Classes | Accuracy, ROC-AUC, F1-Score | All classes equally important |
| Imbalanced Classes | Precision-Recall AUC, F1-Score | Focus on minority class |
| Medical Diagnosis | Recall (Sensitivity) | Missing positives is costly |
| Spam Detection | Precision | False positives annoy users |
| Fraud Detection | Balanced: F1, Recall | Balance catching fraud vs false alarms |

### 🔍 **Practical Guidelines:**

1. **Always examine the confusion matrix first**
2. **Consider the business cost of different error types**
3. **Use multiple metrics for comprehensive evaluation**
4. **Create visualizations to communicate results effectively**
5. **Validate performance on hold-out test sets**

### 🚀 **Next Steps:**
- Practice with different datasets and class imbalances
- Learn about ensemble methods for better performance
- Explore advanced techniques like cost-sensitive learning
- Study cross-validation for robust evaluation